© 2025 Mobile Perception Systems Lab at TU/e. All rights reserved. Licensed under the MIT License.

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
cd /content/drive/MyDrive/AnomalySegmentation_Project/eomt

/content/drive/.shortcut-targets-by-id/1x2LBwdmPjKFuYwX3GTOrlr8KeH9a9SXK/AnomalySegmentation_Project/eomt


In [5]:
!pip install -r requirements.txt

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 5.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.3/131.3 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 90.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 109.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 139.3 MB/s 

In [2]:
import yaml
from lightning import seed_everything
import torch
from torch.nn import functional as F
from torch.amp.autocast_mode import autocast
import matplotlib.pyplot as plt
import numpy as np
import warnings
import importlib

seed_everything(0, verbose=False)

device = 0  # TODO: change to the GPU you want to use
img_idx = 246 # TODO: change to the index of the image you want to visualize
config_path_coco = "configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml"
config_path_cityscapes = "configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"
data_path = "../Cityscapes"  # TODO: change to the dataset directory

with open(config_path_coco, "r") as f:
    config_coco = yaml.safe_load(f)

with open(config_path_cityscapes, "r") as f:
    config_city = yaml.safe_load(f)

def create_mapping(images, ignore_index):
    unique_ids = np.unique(np.concatenate([np.unique(img) for img in images]))
    valid_ids = unique_ids[unique_ids != ignore_index]
    colors = np.array(
        [plt.cm.hsv(i / len(valid_ids))[:3] for i in range(len(valid_ids))]
    )
    mapping = {cid: colors[i] for i, cid in enumerate(valid_ids)}
    mapping[ignore_index] = np.array([0, 0, 0])
    return mapping


def apply_colormap(image, mapping):
    colored_image = np.zeros((*image.shape, 3))
    for cid in np.unique(image):
        colored_image[image == cid] = mapping.get(cid, [0, 0, 0])
    return colored_image

ModuleNotFoundError: No module named 'lightning'

In [ ]:
# Check if CUDA (NVIDIA GPU) is available
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU detected: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("No GPU detected, using CPU.")


## Load dataset

Ensure the dataset files are correctly prepared and placed in the folder specified by `data_path`.

In [ ]:
#CITYSCAPES
data_module_name_city, class_name_city = config_city["data"]["class_path"].rsplit(".", 1)
data_module_city = getattr(importlib.import_module(data_module_name_city), class_name_city)
data_kwargs_city = config_city["data"].get("init_args", {})

data_city = data_module_city(
    path=data_path,
    batch_size=1,
    num_workers=0,
    check_empty_targets=False,
    **data_kwargs_city
)
data_city.setup()


#COCO
data_module_name_coco, class_name_coco = config_coco["data"]["class_path"].rsplit(".", 1)
data_module_coco = getattr(importlib.import_module(data_module_name_coco), class_name_coco)
data_kwargs_coco = config_coco["data"].get("init_args", {})

data_coco_meta = data_module_coco(
    path="",
    batch_size=1,
    num_workers=0,
    check_empty_targets=False,
    **data_kwargs_coco
)

## Load model

In [ ]:
warnings.filterwarnings(
    "ignore",
    message=r".*Attribute 'network' is an instance of `nn\.Module` and is already saved during checkpointing.*",
)

def build_and_load_model(config, data_meta, weights_path):
    # Load encoder
    encoder_cfg = config["model"]["init_args"]["network"]["init_args"]["encoder"]
    encoder_module_name, encoder_class_name = encoder_cfg["class_path"].rsplit(".", 1)
    encoder_cls = getattr(importlib.import_module(encoder_module_name), encoder_class_name)
    encoder = encoder_cls(img_size=data_meta.img_size, **encoder_cfg.get("init_args", {}))

    # Load network
    network_cfg = config["model"]["init_args"]["network"]
    network_module_name, network_class_name = network_cfg["class_path"].rsplit(".", 1)
    network_cls = getattr(importlib.import_module(network_module_name), network_class_name)
    network_kwargs = {k: v for k, v in network_cfg["init_args"].items() if k != "encoder"}

    network = network_cls(
        masked_attn_enabled=False,
        num_classes=data_meta.num_classes,
        encoder=encoder,
        **network_kwargs,
    )

    # Load Lightning module
    lit_module_name, lit_class_name = config["model"]["class_path"].rsplit(".", 1)
    lit_cls = getattr(importlib.import_module(lit_module_name), lit_class_name)
    model_kwargs = {k: v for k, v in config["model"]["init_args"].items() if k != "network"}

    if "stuff_classes" in config["data"].get("init_args", {}):
        model_kwargs["stuff_classes"] = config["data"]["init_args"]["stuff_classes"]

    name = config.get("trainer", {}).get("logger", {}).get("init_args", {}).get("name", "")
    is_dinov3 = "dinov3" in name

    if is_dinov3:
        model_kwargs["ckpt_path"] = weights_path
        model_kwargs["delta_weights"] = True

    model = (
        lit_cls(
            img_size=data_meta.img_size,
            num_classes=data_meta.num_classes,
            network=network,
            **model_kwargs,
        )
        .eval()
        .to(device)
    )

    if not is_dinov3:
        try:
            ckpt = torch.load(weights_path, map_location=device)
            if isinstance(ckpt, dict) and "state_dict" in ckpt:
                state_dict = ckpt["state_dict"]
            else:
                state_dict = ckpt
            model.load_state_dict(state_dict, strict=False)
            print(f"Weights successfully loaded from {weights_path}!")

        except Exception as e:
            print(f"Error loading weights from {weights_path}: {e}")

    return model

#COCO
state_dict_path_coco = "weights/eomt_coco.bin"
print("Initializing COCO Panoptic model...")
model_panoptic = build_and_load_model(config_coco, data_coco_meta, state_dict_path_coco)

#CITYSCAPES
state_dict_path_city = "weights/eomt_cityscapes.bin"
print("Initializing Cityscapes Semantic model...")
model_semantic = build_and_load_model(config_city, data_city, state_dict_path_city)

## Semantic inference (pixel-wise classification)

> This inference method also works when applied to a model trained for panoptic segmentation.

Semantic inference computes per-pixel class scores by combining mask and class predictions:

$$
\sum_i p_i(c) \cdot m_i[h, w]
$$

Here, $p_i(c)$ is the class probability for class $c$ (excluding "no object"), and $m_i[h, w]$ is the sigmoid-normalized mask value for query $i$ at pixel $(h, w)$. The final class is selected by taking the argmax over classes.  
  
*This inference method was originally introduced in MaskFormer.*

In [ ]:
IGNORE_INDEX = 255

def infer_semantic(model_sem, data_val, img, target):
    with torch.no_grad(), autocast(dtype=torch.float16, device_type="cuda"):
        imgs = [img.to(device)]
        img_sizes = [img.shape[-2:] for img in imgs]

        crops, origins = model_sem.window_imgs_semantic(imgs)

        mask_logits_per_layer, class_logits_per_layer = model_sem(crops)

        mask_logits = F.interpolate(
            mask_logits_per_layer[-1], data_val.img_size, mode="bilinear"
        )

        crop_logits = model_sem.to_per_pixel_logits_semantic(
            mask_logits, class_logits_per_layer[-1]
        )
        logits = model_sem.revert_window_logits_semantic(crop_logits, origins, img_sizes)
        preds = logits[0].argmax(0).cpu()

    pred_array = preds.numpy()
    target_array = model_sem.to_per_pixel_targets_semantic([target], IGNORE_INDEX)[
        0
    ].numpy()
    return pred_array, target_array


def plot_semantic_results(img, pred_array, target_array):
    mapping = create_mapping([pred_array, target_array], IGNORE_INDEX)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img.permute(1, 2, 0).cpu().numpy())
    axes[0].set_title("Image")
    axes[1].imshow(apply_colormap(pred_array, mapping))
    axes[1].set_title("Prediction")
    axes[2].imshow(apply_colormap(target_array, mapping))
    axes[2].set_title("Target")

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

img, target = data_city.val_dataloader().dataset[img_idx]
pred_array, target_array = infer_semantic(model_semantic, data_city, img, target)
plot_semantic_results(img, pred_array, target_array)

## Panoptic inference (segmentation with instance IDs)

> This inference method also works when applied to a model trained for instance segmentation.

Panoptic inference assigns each pixel $[h, w]$ to the query $i$ that maximizes the product of class and mask confidence:

$$
p_i(c_i) \cdot m_i[h, w]
$$

where $c_i = \arg\max_c \, p_i(c)$ is the most likely class for query $i$. A pixel is assigned to a query only if both the class confidence and mask confidence are high. Pixels assigned to the same query form a segment labeled with $c_i$. "Stuff" segments with the same class are merged; "thing" segments are kept distinct using the query index. Low-confidence and heavily occluded predictions are filtered out.  
  
*This inference method was originally introduced in MaskFormer.*

In [ ]:
def infer_panoptic(model_pan, img, target):
    with torch.no_grad(), autocast(dtype=torch.float16, device_type="cuda"):
        imgs = [img.to(device)]
        img_sizes = [img.shape[-2:] for img in imgs]

        transformed_imgs = model_pan.resize_and_pad_imgs_instance_panoptic(imgs)
        mask_logits_per_layer, class_logits_per_layer = model_pan(transformed_imgs)
        mask_logits = F.interpolate(
            mask_logits_per_layer[-1], model_pan.img_size, mode="bilinear"
        )
        mask_logits = model_pan.revert_resize_and_pad_logits_instance_panoptic(
            mask_logits, img_sizes
        )

        preds = model_pan.to_per_pixel_preds_panoptic(
            mask_logits,
            class_logits_per_layer[-1],
            model_pan.stuff_classes,
            model_pan.mask_thresh,
            model_pan.overlap_thresh,
        )[0].cpu()

    pred = preds.numpy()
    sem_pred, inst_pred = pred[..., 0], pred[..., 1]

    target_seg = model_pan.to_per_pixel_targets_panoptic([target])[0].cpu().numpy()
    sem_target, inst_target = target_seg[..., 0], target_seg[..., 1]

    return sem_pred, inst_pred, sem_target, inst_target


def draw_black_border(sem, inst, mapping):
    h, w = sem.shape
    out = np.zeros((h, w, 3))
    for s in np.unique(sem):
        out[sem == s] = mapping[s]

    combined = sem.astype(np.int64) * 100000 + inst.astype(np.int64)
    border = np.zeros((h, w), dtype=bool)
    border[1:, :] |= combined[1:, :] != combined[:-1, :]
    border[:-1, :] |= combined[1:, :] != combined[:-1, :]
    border[:, 1:] |= combined[:, 1:] != combined[:, :-1]
    border[:, :-1] |= combined[:, 1:] != combined[:, :-1]
    out[border] = 0
    return out


def plot_panoptic_results(img, sem_pred, inst_pred, sem_target, inst_target, num_classes):
    all_ids = np.union1d(np.unique(sem_pred), np.unique(sem_target))
    mapping = {
        s: (
            [0, 0, 0]
            if s == -1 or s == num_classes
            else plt.cm.hsv(i / len(all_ids))[:3]
        )
        for i, s in enumerate(all_ids)
    }

    vis_pred = draw_black_border(sem_pred, inst_pred, mapping)
    vis_target = draw_black_border(sem_target, inst_target, mapping)

    img_np = (
        img.cpu().numpy().transpose(1, 2, 0) if img.dim() == 3 else img.cpu().numpy()
    )

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img_np)
    axes[0].set_title("Input")
    axes[1].imshow(vis_pred)
    axes[1].set_title("Prediction")
    axes[2].imshow(vis_target)
    axes[2].set_title("Target")

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


img, target = data_city.val_dataloader().dataset[img_idx]
sem_pred, inst_pred, sem_target, inst_target = infer_panoptic(model_panoptic, img, target)
plot_panoptic_results(img, sem_pred, inst_pred, sem_target, inst_target, model_panoptic.num_classes)

## Cityscapes vs Coco on Cityscapes Dataset

In [ ]:
import torch
from torchmetrics.classification import MulticlassJaccardIndex
from tqdm import tqdm

IGNORE_INDEX = 255
SINK_CLASS = 19

metric_city = MulticlassJaccardIndex(
    num_classes=20,
    ignore_index=IGNORE_INDEX,
    average="none"
).to(device)

metric_coco = MulticlassJaccardIndex(
    num_classes=20,
    ignore_index=IGNORE_INDEX,
    average="none"
).to(device)


mapping_tensor = torch.full(
    (256,),
    SINK_CLASS,
    dtype=torch.long,
    device=device
)

mapping_tensor[0] = 11    # person -> person
mapping_tensor[1] = 18    # bicycle -> bicycle
mapping_tensor[2] = 13    # car -> car
mapping_tensor[3] = 17    # motorcycle -> motorcycle

mapping_tensor[5] = 15    # bus -> bus
mapping_tensor[6] = 16    # train -> train
mapping_tensor[7] = 14    # truck -> truck

mapping_tensor[9] = 6     # traffic light -> traffic light
mapping_tensor[11] = 7    # stop sign -> traffic sign

mapping_tensor[58] = 8    # potted plant -> vegetation

mapping_tensor[86] = 2    # door-stuff -> building
mapping_tensor[88] = 8    # flower -> vegetation
mapping_tensor[90] = 9    # gravel -> terrain
mapping_tensor[91] = 2    # house -> building

mapping_tensor[100] = 0   # road -> road
mapping_tensor[101] = 2   # roof -> building
mapping_tensor[102] = 9   # sand -> terrain

mapping_tensor[109] = 3   # wall-brick -> wall
mapping_tensor[110] = 3   # wall-stone -> wall
mapping_tensor[111] = 3   # wall-tile -> wall
mapping_tensor[112] = 3   # wall-wood -> wall

mapping_tensor[114] = 2   # window-blind -> building
mapping_tensor[115] = 2   # window-other -> building

mapping_tensor[116] = 8   # tree-merged -> vegetation
mapping_tensor[117] = 4   # fence-merged -> fence
mapping_tensor[119] = 10  # sky-other-merged -> sky

mapping_tensor[123] = 1   # pavement-merged -> sidewalk
mapping_tensor[125] = 9   # grass-merged -> terrain
mapping_tensor[126] = 9   # dirt-merged -> terrain

mapping_tensor[129] = 2   # building-other-merged -> building
mapping_tensor[130] = 9   # rock-merged -> terrain
mapping_tensor[131] = 3   # wall-other-merged -> wall


# Excluded:
# 5  pole
# 12 rider

class_info = [
    (0, "road"),
    (1, "sidewalk"),
    (2, "building"),
    (3, "wall"),
    (4, "fence"),
    (6, "traffic light"),
    (7, "traffic sign"),
    (8, "vegetation"),
    (9, "terrain"),
    (10, "sky"),
    (11, "person"),
    (13, "car"),
    (14, "truck"),
    (15, "bus"),
    (16, "train"),
    (17, "motorcycle"),
    (18, "bicycle"),
]

common_class_ids = [cid for cid, _ in class_info]
common_class_names = [name for _, name in class_info]

common_class_ids_tensor = torch.tensor(
    common_class_ids,
    dtype=torch.long,
    device=device
)


# 4. EVALUATION LOOP

val_dataloader = data_city.val_dataloader()

model_semantic.eval()
model_panoptic.eval()

print("Starting quantitative common-class mIoU evaluation...")

with torch.inference_mode():
    for batch_idx, batch in enumerate(tqdm(val_dataloader, desc="Processing")):
        batched_imgs, batched_targets = batch

        img = batched_imgs[0]
        target = batched_targets[0]

        # Cityscapes model native prediction
        preds_city_np, target_array_np = infer_semantic(
            model_semantic,
            data_city,
            img,
            target
        )

        preds_city_tensor = torch.as_tensor(
            preds_city_np,
            dtype=torch.long,
            device=device
        )

        target_tensor = torch.as_tensor(
            target_array_np,
            dtype=torch.long,
            device=device
        )

        #remove pole and rider
        target_common_tensor = target_tensor.clone()
        target_common_tensor[
            ~torch.isin(target_common_tensor, common_class_ids_tensor)
        ] = IGNORE_INDEX

        # Update Cityscapes metric
        metric_city.update( preds_city_tensor, target_common_tensor)

        # COCO panoptic model prediction
        preds_coco_np, _ = infer_semantic(
            model_panoptic,
            data_coco_meta,
            img,
            target
        )

        preds_coco_tensor = torch.as_tensor(
            preds_coco_np,
            dtype=torch.long,
            device=device
        )

        # COCO -> Cityscapes mapping
        preds_mapped_tensor = mapping_tensor[preds_coco_tensor]

        # Update COCO-mapped metric
        metric_coco.update(preds_mapped_tensor,target_common_tensor)



#RESULTS
iou_array_city = metric_city.compute()
iou_array_coco = metric_coco.compute()

print("\n" + "=" * 85)
print(f"{'CLASS':<20} | {'Cityscapes IoU (%)':<22} | {'COCO Mapped IoU (%)':<22}")
print("-" * 85)

valid_city_values = []
valid_coco_values = []

for cid, name in class_info:
    city_iou = iou_array_city[cid]
    coco_iou = iou_array_coco[cid]

    if torch.isnan(city_iou):
        city_str = f"{'N/A':>18}"
    else:
        city_val = city_iou.item() * 100
        city_str = f"{city_val:>18.2f}"
        valid_city_values.append(city_iou)

    if torch.isnan(coco_iou):
        coco_str = f"{'N/A':>18}"
    else:
        coco_val = coco_iou.item() * 100
        coco_str = f"{coco_val:>18.2f}"
        valid_coco_values.append(coco_iou)

    print(f"{name:<20} | {city_str} | {coco_str}")


# 6. FINAL COMMON-CLASS mIoU


valid_city_tensor = torch.stack(valid_city_values) if len(valid_city_values) > 0 else None
valid_coco_tensor = torch.stack(valid_coco_values) if len(valid_coco_values) > 0 else None

miou_city = valid_city_tensor.mean().item() * 100 if valid_city_tensor is not None else 0.0
miou_coco = valid_coco_tensor.mean().item() * 100 if valid_coco_tensor is not None else 0.0

print("-" * 85)
print(f"{'TOTAL common-17 mIoU':<20} | {miou_city:>18.2f} | {miou_coco:>18.2f}")
print("=" * 85)

metric_city.reset()
metric_coco.reset()

In [ ]:
!mkdir -p /content/Cityscapes_local
!cp /content/drive/MyDrive/AnomalySegmentation_Project/Cityscapes/*.zip /content/Cityscapes_local/

In [ ]:
!wandb login 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

Finetune only Head

In [ ]:
!python main.py fit \
  -c configs/dinov2/cityscapes/semantic/eomt_finetune_base_640.yaml \
  --data.batch_size 8 \
  --trainer.accumulate_grad_batches 2 \
  --data.path /content/Cityscapes_local \
  --trainer.devices 1 \
  --trainer.logger.init_args.name "finetune_only_Head_batch16_final" \
  --trainer.logger.init_args.version "finetune_only_Head_batch16_final" \
  --model.network.init_args.unfreeze_last_n_blocks 0 \
  --model.load_ckpt_class_head False \
  --model.ckpt_path "./weights/eomt_coco.bin"

Finetune Unfreeze 1

In [ ]:
!python main.py fit \
  -c configs/dinov2/cityscapes/semantic/eomt_finetune_base_640.yaml \
  --data.batch_size 8   \
  --trainer.accumulate_grad_batches 2 \
  --data.path /content/Cityscapes_local \
  --trainer.devices 1 \
  --trainer.logger.init_args.name "finetune_unfreeze1_batch16_final" \
  --trainer.logger.init_args.version "finetune_unfreeze1_batch16_final" \
  --model.network.init_args.unfreeze_last_n_blocks 1 \
  --model.load_ckpt_class_head False \
  --model.ckpt_path "./weights/eomt_coco.bin"

Finetune Unfreeze 2 blocks

In [ ]:
!python main.py fit \
  -c configs/dinov2/cityscapes/semantic/eomt_finetune_base_640.yaml \
  --data.batch_size 8 \
  --trainer.accumulate_grad_batches 2 \
  --data.path /content/Cityscapes_local \
  --trainer.devices 1 \
  --trainer.logger.init_args.name "finetune_unfreeze2_batch16_final" \
  --trainer.logger.init_args.version "finetune_unfreeze2_batch16_final" \
  --model.network.init_args.unfreeze_last_n_blocks 2 \
  --model.load_ckpt_class_head False \
  --model.ckpt_path "./weights/eomt_coco.bin"

## Coco vs Finetuned Coco


In [ ]:
with open("configs/dinov2/cityscapes/semantic/eomt_finetune_base_640.yaml", "r") as f:
    config_city_finetune = yaml.safe_load(f)

data_module_name_city_finetune, class_name_city_finetune = config_city_finetune["data"]["class_path"].rsplit(".", 1)
data_module_city_finetune = getattr(importlib.import_module(data_module_name_city_finetune), class_name_city_finetune)
data_kwargs_city_finetune = config_city_finetune["data"].get("init_args", {})

data_city_finetune = data_module_city_finetune(
    path=data_path,
    batch_size=1,
    num_workers=0,
    check_empty_targets=False,
    **data_kwargs_city_finetune
)
data_city_finetune.setup()

In [ ]:
from torchmetrics.classification import MulticlassJaccardIndex
from tqdm import tqdm
import torch

IGNORE_INDEX = 255
SINK_CLASS = 19

state_dict_path_ft = "eomt/finetune_unfreeze2_batch16_final/checkpoints/best.ckpt"

print("Initializing Fine-Tuned model...")
model_ft = build_and_load_model(
    config_city_finetune,
    data_city_finetune,
    state_dict_path_ft
)

metric_coco_old = MulticlassJaccardIndex(
    num_classes=20,
    ignore_index=IGNORE_INDEX,
    average="none"
).to(device)

metric_ft_prog = MulticlassJaccardIndex(
    num_classes=20,
    ignore_index=IGNORE_INDEX,
    average="none"
).to(device)



mapping_tensor = torch.full(
    (256,),
    SINK_CLASS,
    dtype=torch.long,
    device=device
)

# Things
mapping_tensor[0] = 11; mapping_tensor[1] = 18; mapping_tensor[2] = 13; mapping_tensor[3] = 17
mapping_tensor[5] = 15; mapping_tensor[6] = 16; mapping_tensor[7] = 14; mapping_tensor[9] = 6
mapping_tensor[11] = 7; mapping_tensor[58] = 8

# Stuff
mapping_tensor[86] = 2; mapping_tensor[88] = 8; mapping_tensor[90] = 9; mapping_tensor[91] = 2
mapping_tensor[100] = 0; mapping_tensor[101] = 2; mapping_tensor[102] = 9
mapping_tensor[109] = 3; mapping_tensor[110] = 3; mapping_tensor[111] = 3; mapping_tensor[112] = 3
mapping_tensor[114] = 2; mapping_tensor[115] = 2; mapping_tensor[116] = 8; mapping_tensor[117] = 4
mapping_tensor[119] = 10; mapping_tensor[123] = 1; mapping_tensor[125] = 9; mapping_tensor[126] = 9
mapping_tensor[129] = 2; mapping_tensor[130] = 9; mapping_tensor[131] = 3


class_info = [
    (0, "road"),
    (1, "sidewalk"),
    (2, "building"),
    (3, "wall"),
    (4, "fence"),
    (6, "traffic light"),
    (7, "traffic sign"),
    (8, "vegetation"),
    (9, "terrain"),
    (10, "sky"),
    (11, "person"),
    (13, "car"),
    (14, "truck"),
    (15, "bus"),
    (16, "train"),
    (17, "motorcycle"),
    (18, "bicycle"),
]

common_class_ids = [cid for cid, _ in class_info]

common_class_ids_tensor = torch.tensor(
    common_class_ids,
    dtype=torch.long,
    device=device
)


# EVALUATION LOOP

val_dataloader = data_city_finetune.val_dataloader()

model_panoptic.eval()
model_ft.eval()

print("Confronto: COCO Mapped vs COCO Fine-Tuned")

with torch.inference_mode():
    for batch in tqdm(val_dataloader, desc="Processing"):

        batched_imgs, batched_targets = batch

        img = batched_imgs[0]
        target = batched_targets[0]

        preds_ft_np, target_array_np = infer_semantic(
            model_ft,
            data_city_finetune,
            img,
            target
        )

        preds_ft_tensor = torch.as_tensor(
            preds_ft_np,
            dtype=torch.long,
            device=device
        )

        target_tensor = torch.as_tensor(
            target_array_np,
            dtype=torch.long,
            device=device
        )

        target_common_tensor = target_tensor.clone()
        target_common_tensor[
            ~torch.isin(target_common_tensor, common_class_ids_tensor)
        ] = IGNORE_INDEX

        preds_coco_np, _ = infer_semantic(
            model_panoptic,
            data_coco_meta,
            img,
            target
        )

        preds_coco_tensor = torch.as_tensor(
            preds_coco_np,
            dtype=torch.long,
            device=device
        )

        preds_mapped_tensor = mapping_tensor[preds_coco_tensor]

        metric_coco_old.update(
            preds_mapped_tensor,
            target_common_tensor
        )

        metric_ft_prog.update(
            preds_ft_tensor,
            target_common_tensor
        )


# RESULTS
iou_array_old = metric_coco_old.compute()
iou_array_ft = metric_ft_prog.compute()

print("\n" + "=" * 85)
print(f"{'CLASS':<20} | {'COCO Mapped IoU (%)':<22} | {'Fine-Tuned IoU (%)':<22}")
print("-" * 85)

valid_old_values = []
valid_ft_values = []

for cid, name in class_info:
    old_iou = iou_array_old[cid]
    ft_iou = iou_array_ft[cid]

    if torch.isnan(old_iou):
        old_str = f"{'N/A':>18}"
    else:
        old_str = f"{old_iou.item() * 100:>18.2f}"
        valid_old_values.append(old_iou)

    if torch.isnan(ft_iou):
        ft_str = f"{'N/A':>18}"
    else:
        ft_str = f"{ft_iou.item() * 100:>18.2f}"
        valid_ft_values.append(ft_iou)

    print(f"{name:<20} | {old_str} | {ft_str}")


valid_old_tensor = torch.stack(valid_old_values) if len(valid_old_values) > 0 else None
valid_ft_tensor = torch.stack(valid_ft_values) if len(valid_ft_values) > 0 else None

miou_old = valid_old_tensor.mean().item() * 100 if valid_old_tensor is not None else 0.0
miou_ft = valid_ft_tensor.mean().item() * 100 if valid_ft_tensor is not None else 0.0

print("-" * 85)
print(f"{'TOTAL common-17 mIoU':<20} | {miou_old:>18.2f} | {miou_ft:>18.2f}")
print("=" * 85)

metric_coco_old.reset()
metric_ft_prog.reset()

## Cityscapes vs coco-finetuned

In [ ]:
from torchmetrics.classification import MulticlassJaccardIndex
from tqdm import tqdm
import torch

IGNORE_INDEX = 255

state_dict_path_ft = "eomt/finetune_unfreeze2_batch16_final/checkpoints/best.ckpt"
print("Initializing fine-tuned model...")
model_ft = build_and_load_model(config_city_finetune, data_city_finetune, state_dict_path_ft)

metric_city_final = MulticlassJaccardIndex(num_classes=19, ignore_index=IGNORE_INDEX, average="none").to(device)
metric_ft_final = MulticlassJaccardIndex(num_classes=19, ignore_index=IGNORE_INDEX, average="none").to(device)

val_dataloader = data_city.val_dataloader()

model_semantic.eval()
model_ft.eval()

print("Starting final comparison: native Cityscapes model vs COCO fine-tuned model (all 19 classes)...")

with torch.inference_mode():
    for batch in tqdm(val_dataloader, desc="Evaluating"):
        batched_imgs, batched_targets = batch
        img = batched_imgs[0]
        target = batched_targets[0]

        preds_city_np, target_array_np = infer_semantic(model_semantic, data_city, img, target)
        preds_city_tensor = torch.as_tensor(preds_city_np, dtype=torch.long, device=device)
        target_tensor = torch.as_tensor(target_array_np, dtype=torch.long, device=device)
        metric_city_final.update(preds_city_tensor, target_tensor)

        preds_ft_np, _ = infer_semantic(model_ft, data_city_finetune, img, target)
        preds_ft_tensor = torch.as_tensor(preds_ft_np, dtype=torch.long, device=device)
        metric_ft_final.update(preds_ft_tensor, target_tensor)

iou_array_city = metric_city_final.compute()
iou_array_ft = metric_ft_final.compute()

all_19_classes = [
    (0, "road"), (1, "sidewalk"), (2, "building"), (3, "wall"), (4, "fence"),
    (5, "pole"), (6, "traffic light"), (7, "traffic sign"), (8, "vegetation"),
    (9, "terrain"), (10, "sky"), (11, "person"), (12, "rider"), (13, "car"),
    (14, "truck"), (15, "bus"), (16, "train"), (17, "motorcycle"), (18, "bicycle")
]

print("\n" + "=" * 85)
print(f"{'CLASS':<18} | {'Cityscapes (%)':<28} | {'COCO Fine-Tuned (%)':<28}")
print("-" * 85)

valid_city_values = []
valid_ft_values = []

for cid, name in all_19_classes:
    city_iou = iou_array_city[cid]
    ft_iou = iou_array_ft[cid]

    if torch.isnan(city_iou):
        city_str = f"{'N/A':>24}"
    else:
        city_str = f"{city_iou.item() * 100:>24.2f}"
        valid_city_values.append(city_iou)

    if torch.isnan(ft_iou):
        ft_str = f"{'N/A':>24}"
    else:
        ft_str = f"{ft_iou.item() * 100:>24.2f}"
        valid_ft_values.append(ft_iou)

    print(f"{name:<18} | {city_str} | {ft_str}")

miou_city = torch.stack(valid_city_values).mean().item() * 100 if valid_city_values else 0.0
miou_ft = torch.stack(valid_ft_values).mean().item() * 100 if valid_ft_values else 0.0

print("-" * 85)
print(f"{'Total mIoU':<18} | {miou_city:>24.2f} | {miou_ft:>24.2f}")
print("=" * 85)

metric_city_final.reset()
metric_ft_final.reset()

## Coco finetuned Only Head vs Unfreeze 1 vs Unfreeze 2

In [ ]:
from torchmetrics.classification import MulticlassJaccardIndex
from tqdm import tqdm
import torch

IGNORE_INDEX = 255
NUM_CLASSES = 19

checkpoint_paths = {
    "Only Head": "eomt/finetune_only_Head_batch16_final/checkpoints/best-checkpoint-epoch=19-metrics/val_iou_all=0.7301.ckpt",
    "Unfreeze 1": "eomt/finetune_unfreeze1_batch16_final/checkpoints/best.ckpt",
    "Unfreeze 2": "eomt/finetune_unfreeze2_batch16_final/checkpoints/best.ckpt"
}

models = {}
metrics = {}

for name, ckpt_path in checkpoint_paths.items():
    print(f"Initializing fine-tuned model: {name}...")
    models[name] = build_and_load_model(config_city_finetune, data_city_finetune, ckpt_path)
    models[name].eval()
    metrics[name] = MulticlassJaccardIndex(num_classes=NUM_CLASSES, ignore_index=IGNORE_INDEX, average="none").to(device)

val_dataloader = data_city_finetune.val_dataloader()

cityscapes_classes = [
    (0, "road"), (1, "sidewalk"), (2, "building"), (3, "wall"), (4, "fence"),
    (5, "pole"), (6, "traffic light"), (7, "traffic sign"), (8, "vegetation"),
    (9, "terrain"), (10, "sky"), (11, "person"), (12, "rider"), (13, "car"),
    (14, "truck"), (15, "bus"), (16, "train"), (17, "motorcycle"), (18, "bicycle")
]

print("Starting final comparison: Only Head vs Unfreeze 1 vs Unfreeze 2")

with torch.inference_mode():
    for batch in tqdm(val_dataloader, desc="Running validation"):
        batched_imgs, batched_targets = batch
        img = batched_imgs[0]
        target = batched_targets[0]

        target_tensor = None

        for name, model in models.items():
            preds_np, target_array_np = infer_semantic(model, data_city_finetune, img, target)
            preds_tensor = torch.as_tensor(preds_np, dtype=torch.long, device=device)

            if target_tensor is None:
                target_tensor = torch.as_tensor(target_array_np, dtype=torch.long, device=device)

            metrics[name].update(preds_tensor, target_tensor)

iou_arrays = {name: metric.compute() for name, metric in metrics.items()}

print("\n" + "=" * 110)
print(f"{'CLASS':<18} | {'Only Head (%)':<24} | {'Unfreeze 1 (%)':<24} | {'Unfreeze 2 (%)':<24}")
print("-" * 110)

valid_values = {name: [] for name in metrics.keys()}

for cid, class_name in cityscapes_classes:
    row = f"{class_name:<18}"

    for name in checkpoint_paths.keys():
        iou = iou_arrays[name][cid]

        if torch.isnan(iou):
            value_str = f"{'N/A':>20}"
        else:
            value_str = f"{iou.item() * 100:>20.2f}"
            valid_values[name].append(iou)

        row += f" | {value_str}"

    print(row)

miou_values = {name: torch.stack(values).mean().item() * 100 if values else 0.0 for name, values in valid_values.items()}

print("-" * 110)
print(f"{'TOTAL mIoU':<18} | {miou_values['Only Head']:>20.2f} | {miou_values['Unfreeze 1']:>20.2f} | {miou_values['Unfreeze 2']:>20.2f}")
print("=" * 110)

for metric in metrics.values():
    metric.reset()

**Exercise 7**

In [6]:
cd /content/drive/MyDrive/AnomalySegmentation_Project/eval

/content/drive/.shortcut-targets-by-id/1x2LBwdmPjKFuYwX3GTOrlr8KeH9a9SXK/AnomalySegmentation_Project/eval


In [ ]:
!python eval_iou.py --loadWeights erfnet_pretrained.pth --datadir /content/Cityscapes/

Loading model: ../trained_models/erfnet.py
Loading weights: ../trained_models/erfnet_pretrained.pth
Model and weights LOADED successfully
Error: datadir could not be loaded
/content/Cityscapes/leftImg8bit/val /content/Cityscapes/gtFine/val
---------------------------------------
Took  0.06556344032287598 seconds
Per-Class IoU:
0.00 Road
0.00 sidewalk
0.00 building
0.00 wall
0.00 fence
0.00 pole
0.00 traffic light
0.00 traffic sign
0.00 vegetation
0.00 terrain
0.00 sky
0.00 person
0.00 rider
0.00 car
0.00 truck
0.00 bus
0.00 train
0.00 motorcycle
0.00 bicycle
MEAN IoU:  0.00 %


In [8]:
!python evalAnomaly.py --model_type erfnet --weights ../trained_models/erfnet_pretrained.pth --input "../Anomaly_Validation_Datasets/Validation_Dataset/RoadAnomaly21/images/*.png"

!python evalAnomaly.py --model_type erfnet --weights ../trained_models/erfnet_pretrained.pth --input "../Anomaly_Validation_Datasets/Validation_Dataset/RoadObsticle21/images/*.webp"

!python evalAnomaly.py --model_type erfnet --weights ../trained_models/erfnet_pretrained.pth --input "../Anomaly_Validation_Datasets/Validation_Dataset/FS_LostFound_full/images/*.png"

!python evalAnomaly.py --model_type erfnet --weights ../trained_models/erfnet_pretrained.pth --input "../Anomaly_Validation_Datasets/Validation_Dataset/fs_static/images/*.jpg"

!python evalAnomaly.py --model_type erfnet --weights ../trained_models/erfnet_pretrained.pth --input "../Anomaly_Validation_Datasets/Validation_Dataset/RoadAnomaly/images/*.jpg"




Initializing ERFNET model...

Starting evaluation loop...
--- ERFNET EVALUATION RESULTS on RoadAnomaly21 (Temperature: 1.0) ---
MSP          -> AUPRC: 29.10 | FPR@TPR95: 62.55
MaxLogit     -> AUPRC: 38.32 | FPR@TPR95: 59.34
MaxEntropy   -> AUPRC: 30.97 | FPR@TPR95: 62.66

Initializing ERFNET model...

Starting evaluation loop...
--- ERFNET EVALUATION RESULTS on RoadObsticle21 (Temperature: 1.0) ---
MSP          -> AUPRC: 2.71 | FPR@TPR95: 65.22
MaxLogit     -> AUPRC: 4.63 | FPR@TPR95: 48.44
MaxEntropy   -> AUPRC: 3.04 | FPR@TPR95: 65.91

Initializing ERFNET model...

Starting evaluation loop...
--- ERFNET EVALUATION RESULTS on FS_LostFound_full (Temperature: 1.0) ---
MSP          -> AUPRC: 1.75 | FPR@TPR95: 50.60
MaxLogit     -> AUPRC: 3.30 | FPR@TPR95: 45.49
MaxEntropy   -> AUPRC: 2.58 | FPR@TPR95: 50.16

Initializing ERFNET model...

Starting evaluation loop...
--- ERFNET EVALUATION RESULTS on fs_static (Temperature: 1.0) ---
MSP          -> AUPRC: 7.47 | FPR@TPR95: 41.84
MaxLogit  

**Exercise 8**

COCO

In [ ]:
!python evalAnomaly.py --model_type eomt --config ../eomt/configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml --weights ../eomt/weights/eomt_coco.bin --input "../Anomaly_Validation_Datasets/Validation_Dataset/RoadAnomaly21/images/*.png"

!python evalAnomaly.py --model_type eomt --config ../eomt/configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml --weights ../eomt/weights/eomt_coco.bin --input "../Anomaly_Validation_Datasets/Validation_Dataset/RoadObsticle21/images/*.webp"

!python evalAnomaly.py --model_type eomt --config ../eomt/configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml --weights ../eomt/weights/eomt_coco.bin --input "../Anomaly_Validation_Datasets/Validation_Dataset/FS_LostFound_full/images/*.png"

!python evalAnomaly.py --model_type eomt --config ../eomt/configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml --weights ../eomt/weights/eomt_coco.bin --input "../Anomaly_Validation_Datasets/Validation_Dataset/fs_static/images/*.jpg"

!python evalAnomaly.py --model_type eomt --config ../eomt/configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml --weights ../eomt/weights/eomt_coco.bin --input "../Anomaly_Validation_Datasets/Validation_Dataset/RoadAnomaly/images/*.jpg"


Initializing EOMT model...
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['network'])`.

Starting evaluation loop...
--- EOMT EVALUATION RESULTS on RoadAnomaly21 (Temperature: 1.0) ---
MSP          -> AUPRC: 16.12 | FPR@TPR95: 84.23
MaxLogit     -> AUPRC: 15.55 | FPR@TPR95: 84.11
MaxEntropy   -> AUPRC: 15.49 | FPR@TPR95: 95.37
RbA          -> AUPRC: 12.24 | FPR@TPR95: 92.89

Initializing EOMT model...
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['network'])`.

Starting evaluation loop...
--- EOMT EVALUATION RESULTS on RoadObsticle21 (Temperature: 1.0) ---
MSP          -> AUPRC: 0.44 

CItyscapes

In [ ]:
!python evalAnomaly.py --model_type eomt --config ../eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml --weights ../eomt/weights/eomt_cityscapes.bin --input "../Anomaly_Validation_Datasets/Validation_Dataset/RoadAnomaly21/images/*.png"

!python evalAnomaly.py --model_type eomt --config ../eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml --weights ../eomt/weights/eomt_cityscapes.bin --input "../Anomaly_Validation_Datasets/Validation_Dataset/RoadObsticle21/images/*.webp"

!python evalAnomaly.py --model_type eomt --config ../eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml --weights ../eomt/weights/eomt_cityscapes.bin --input "../Anomaly_Validation_Datasets/Validation_Dataset/FS_LostFound_full/images/*.png"

!python evalAnomaly.py --model_type eomt --config ../eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml --weights ../eomt/weights/eomt_cityscapes.bin --input "../Anomaly_Validation_Datasets/Validation_Dataset/fs_static/images/*.jpg"

!python evalAnomaly.py --model_type eomt --config ../eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml --weights ../eomt/weights/eomt_cityscapes.bin --input "../Anomaly_Validation_Datasets/Validation_Dataset/RoadAnomaly/images/*.jpg"


Initializing EOMT model...
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['network'])`.

Starting evaluation loop...
--- EOMT EVALUATION RESULTS on RoadAnomaly21 (Temperature: 1.0) ---
MSP          -> AUPRC: 12.15 | FPR@TPR95: 94.81
MaxLogit     -> AUPRC: 12.25 | FPR@TPR95: 96.18
MaxEntropy   -> AUPRC: 12.22 | FPR@TPR95: 94.80
RbA          -> AUPRC: 19.69 | FPR@TPR95: 88.06

Initializing EOMT model...
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['network'])`.

Starting evaluation loop...
--- EOMT EVALUATION RESULTS on RoadObsticle21 (Temperature: 1.0) ---
MSP          -> AUPRC: 0.36

In [7]:
!python evalAnomaly.py --model_type eomt --config ../eomt/configs/dinov2/cityscapes/semantic/eomt_finetune_base_640.yaml --weights ../eomt/eomt/finetune_unfreeze2_batch16_final/checkpoints/best.ckpt --input "../Anomaly_Validation_Datasets/Validation_Dataset/RoadAnomaly21/images/*.png"

!python evalAnomaly.py --model_type eomt --config ../eomt/configs/dinov2/cityscapes/semantic/eomt_finetune_base_640.yaml --weights ../eomt/eomt/finetune_unfreeze2_batch16_final/checkpoints/best.ckpt --input "../Anomaly_Validation_Datasets/Validation_Dataset/RoadObsticle21/images/*.webp"

!python evalAnomaly.py --model_type eomt --config ../eomt/configs/dinov2/cityscapes/semantic/eomt_finetune_base_640.yaml --weights ../eomt/eomt/finetune_unfreeze2_batch16_final/checkpoints/best.ckpt --input "../Anomaly_Validation_Datasets/Validation_Dataset/FS_LostFound_full/images/*.png"

!python evalAnomaly.py --model_type eomt --config ../eomt/configs/dinov2/cityscapes/semantic/eomt_finetune_base_640.yaml --weights ../eomt/eomt/finetune_unfreeze2_batch16_final/checkpoints/best.ckpt --input "../Anomaly_Validation_Datasets/Validation_Dataset/fs_static/images/*.jpg"

!python evalAnomaly.py --model_type eomt --config ../eomt/configs/dinov2/cityscapes/semantic/eomt_finetune_base_640.yaml --weights ../eomt/eomt/finetune_unfreeze2_batch16_final/checkpoints/best.ckpt --input "../Anomaly_Validation_Datasets/Validation_Dataset/RoadAnomaly/images/*.jpg"


Initializing EOMT model...
model.safetensors: 100% 346M/346M [00:04<00:00, 84.3MB/s]
2026-05-15 08:50:23.415742: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['network'])`.

Starting evaluation loop...
--- EOMT EVALUATION RESULTS on RoadAnomaly21 (Temperature: 1.0) ---
MSP          -> AUPRC: 17.39 | FPR@TPR95: 88.36
MaxLogit     -> AUPRC: 11.34 | FPR@TPR95: 93.08
MaxEntropy   -> AUPRC: 14.83 | FPR@TPR95: 99.89
RbA          -> AUPRC: 11.26 | FPR@TPR95: 93.63

Initializing EOMT model...
2026-05-15 0